In [1]:
import sys
from pathlib import Path

sys.path.append(Path.cwd().parent.parent.as_posix())
import pickle

path = "/mnt/labstore/bespoke_olap/cache/validate/8f780a048c94246fdec8bc9732e859469ddc6b27cd7b006c71d0c1927525c0b5.pkl"

with open(path.strip(), "rb") as f:
    data = pickle.load(f)

print(f"Type: {type(data)}")

values = data.__dict__
for key, value in values.items():
    print(f"{key} -> {value}")

Type: <class 'pipeline.tools.validate.validate_cache_type.ValidateCacheType'>
outputs -> Result columns do not match for query 15 with placeholders: {'DATE': '1995-09-01', 'STREAM_ID': '2'}
(SQL: with revenue (supplier_no, total_revenue) as (
    select
        l_suppkey,
        sum(l_extendedprice * (1 - l_discount))
    from
        lineitem
    where
        l_shipdate >= date '1995-09-01'
        and l_shipdate < date '1995-09-01' + interval '3' month
    group by
        l_suppkey
)
select
    s_suppkey,
    s_name,
    s_address,
    s_phone,
    total_revenue
from
    supplier,
    revenue
where
    s_suppkey = supplier_no
    and total_revenue = (
        select
            max(total_revenue)
        from
            revenue
    )
order by
    s_suppkey;)
DuckDB columns: ['s_suppkey', 's_name', 's_address', 's_phone', 'total_revenue']
Implementation columns: ['promo_revenue']
Build time (ms): 24232.1

metrics -> {'validation/scale_factor': 20, 'validation/correct': False, 'val

# Delete this entry

In [ ]:
# delete the git commit
# extract snapshot hash from cache path
snapshot_name = Path(path).stem
print(f"Snapshot name: {snapshot_name}")

# ask for user confirmation
confirm = input(f"Do you want to delete this cache entry? (y/n) {snapshot_name}")
if confirm.lower() == "y":
    # snapshot_hash = values.get("snapshot_hash")
    # if snapshot_hash:
    cmd1 = f"git -C /home/jwehrstein/bespoke_olap/output update-ref -d refs/snapshots/snapshot-{snapshot_name}"
    cmd2 = f"git -C /home/jwehrstein/bespoke_olap/output push cache_repo --delete refs/snapshots/snapshot-{snapshot_name}"

    try:
        import subprocess

        subprocess.run(cmd1, shell=True, check=True)
        subprocess.run(cmd2, shell=True, check=True)
        print("Git snapshot deleted successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Error deleting git snapshot: {e}")
        raise e

    # delete the file
    try:
        Path(path).unlink()
        print("Cache entry deleted successfully.")
    except Exception as e:
        print(f"Error deleting cache entry: {e}")
else:
    print("Deletion cancelled by user.")

Snapshot name: 8f780a048c94246fdec8bc9732e859469ddc6b27cd7b006c71d0c1927525c0b5
Git snapshot deleted successfully.
Cache entry deleted successfully.


remote: warning: deleting a non-existent ref        
To git://c01/bespoke_cache.git
 - [deleted]             refs/snapshots/snapshot-8f780a048c94246fdec8bc9732e859469ddc6b27cd7b006c71d0c1927525c0b5
